<a href="https://colab.research.google.com/github/BatmanGudu007/Ybi-foundation-Internship/blob/main/ANN_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANN Classification

### Steps to Build an ANN in Python

Building an Artificial Neural Network (ANN) typically involves the following key steps:

1. **Data Preprocessing**:
   - Load the dataset.
   - Handle missing values.
   - Encode categorical variables (e.g., One-Hot Encoding).
   - Split the dataset into Training and Testing sets.
   - **Feature Scaling**: Essential for neural networks so that features with larger ranges do not dominate the training process.

2. **Building the ANN**:
   - Initialize the network as a Sequential model.
   - Add the input layer and the first hidden layer (defining the number of neurons and the activation function, usually ReLU).
   - Add extra hidden layers if needed.
   - Add the output layer (using Sigmoid for binary classification, Softmax for multi-class classification, or no activation for regression).

3. **Compiling the ANN**:
   - Choose an optimizer (e.g., Adam or SGD).
   - Choose a loss function (e.g., `binary_crossentropy` for binary classification).
   - Choose evaluation metrics (e.g., `accuracy`).

4. **Training the ANN**:
   - Fit the model to the training dataset using a specified number of epochs and batch size.

5. **Evaluation and Predictions**:
   - Predict results on the test set.
   - Evaluate performance using metrics like accuracy, a confusion matrix, or classification report.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

# 1. Generate dummy classification data & Preprocess
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. Build the ANN
model = keras.Sequential([
    # Input layer & First Hidden Layer
    keras.layers.Dense(units=16, activation='relu', input_shape=(10,)),
    # Second Hidden Layer
    keras.layers.Dense(units=8, activation='relu'),
    # Output Layer (Binary Classification)
    keras.layers.Dense(units=1, activation='sigmoid')
])

# 3. Compile the ANN
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Train the ANN
history = model.fit(X_train, y_train, batch_size=32, epochs=20, validation_split=0.1, verbose=1)

# 5. Evaluate the ANN
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {accuracy:.4f}")


### Explanation of Activation Functions Used in this Model

#### 1. ReLU (Rectified Linear Unit)
* **Where it is used:** Hidden layers (`units=16` and `units=8`).
* **Mathematical formula:** $f(x) = \max(0, x)$
* **How it works:**
  - If the input is positive, it outputs the input directly.
  - If the input is negative, it outputs zero.
* **Why we use it:**
  - **Introduces Non-linearity:** Without non-linear activation functions, a multi-layer neural network would behave just like a single-layer linear regression model, regardless of how many hidden layers you add.
  - **Computational Efficiency:** It is extremely fast to compute compared to older functions like Sigmoid or Tanh.
  - **Reduces Vanishing Gradient Problem:** For positive inputs, the gradient (derivative) is always constant ($1$), which helps the network train faster and avoid getting stuck during backpropagation.

#### 2. Sigmoid
* **Where it is used:** Output layer (`units=1` for binary classification).
* **Mathematical formula:** $f(x) = \frac{1}{1 + e^{-x}}$
* **How it works:**
  - It maps any real-valued number into a strict range between **0 and 1**.
  - Extremely large positive values map close to 1, while extremely negative values map close to 0.
* **Why we use it:**
  - Since this is a binary classification task (predicting whether an instance belongs to Class 0 or Class 1), the Sigmoid function allows the network's final output to be interpreted directly as a **probability** (e.g., a prediction of $0.85$ means an $85\%$ probability of being Class 1).

### Improving Performance with Dropout Layers

**Dropout** is a regularization technique used to prevent **overfitting** in neural networks.

* **How it works:** During training, Dropout randomly ignores (or "drops out") a specified fraction of neurons (e.g., $20\%$) at each update step. This forces the remaining neurons to learn more robust features rather than co-adapting and relying too heavily on specific neighbors.
* **During Testing:** Dropout is deactivated automatically during evaluation and prediction, so the full power of the network is utilized.

Here is how you can incorporate Dropout layers into your sequential model structure, placed typically after your dense hidden layers.

In [ ]:
# Rebuilding the model with Dropout layers to prevent overfitting
model_with_dropout = keras.Sequential([
    # Input layer & First Hidden Layer
    keras.layers.Dense(units=16, activation='relu', input_shape=(10,)),
    # Dropout 20% of the neurons from the first hidden layer
    keras.layers.Dropout(0.2),

    # Second Hidden Layer
    keras.layers.Dense(units=8, activation='relu'),
    # Dropout 20% of the neurons from the second hidden layer
    keras.layers.Dropout(0.2),

    # Output Layer
    keras.layers.Dense(units=1, activation='sigmoid')
])

# Compile the new model
model_with_dropout.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history_dropout = model_with_dropout.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=20,
    validation_split=0.1,
    verbose=1
)

# Evaluate
loss_dp, accuracy_dp = model_with_dropout.evaluate(X_test, y_test)
print(f"\nTest Accuracy with Dropout: {accuracy_dp:.4f}")

### Comparing Baseline vs. Dropout Model
Let's compare the evaluation results and learning curves of both models to see how the dropout regularization affected the training and validation performance.

In [ ]:
import matplotlib.pyplot as plt

# Compare final test evaluation metrics
print("=== Performance Comparison ===")
print(f"Baseline Model - Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")
print(f"Dropout Model  - Test Loss: {loss_dp:.4f}, Test Accuracy: {accuracy_dp:.4f}")

# Plot learning curves
plt.figure(figsize=(12, 5))

# Plot training & validation accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['val_accuracy'], label='Baseline Val Acc', linestyle='--')
plt.plot(history_dropout.history['val_accuracy'], label='Dropout Val Acc')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot training & validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['val_loss'], label='Baseline Val Loss', linestyle='--')
plt.plot(history_dropout.history['val_loss'], label='Dropout Val Loss')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Advanced Techniques to Improve ANN Performance

Beyond adding Dropout, here are several powerful strategies to optimize your neural network:

1. **Batch Normalization**:
   - Normalizes the inputs of each layer so that they have a mean of 0 and a variance of 1. This stabilizes the learning process, allows higher learning rates, and often accelerates training.

2. **Early Stopping**:
   - Monitors a validation metric (such as validation loss) and stops training when the performance stops improving. This prevents unnecessary epochs and limits overfitting.

3. **Learning Rate Scheduling (Learning Rate Decay)**:
   - Starts with a higher learning rate to find the global minimum region quickly, and then progressively reduces the learning rate to fine-tune the weights without overshooting.

4. **Hyperparameter Tuning (Keras Tuner)**:
   - Systematically searches for the best parameters (e.g., number of hidden layers, number of neurons per layer, choice of optimizer, and activation functions).

Let's implement **Batch Normalization** and **Early Stopping** in our Keras model:

In [ ]:
# 1. Define Early Stopping Callback
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,             # Stop training if validation loss doesn't improve for 5 consecutive epochs
    restore_best_weights=True # Revert back to weights from the epoch with the lowest validation loss
)

# 2. Build Model incorporating Batch Normalization and Dropout
optimized_model = keras.Sequential([
    # Input layer + Dense
    keras.layers.Dense(units=32, input_shape=(10,)),
    keras.layers.BatchNormalization(), # Normalize activations before activation function
    keras.layers.Activation('relu'),
    keras.layers.Dropout(0.2),

    # Second Hidden Layer
    keras.layers.Dense(units=16),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dropout(0.2),

    # Output Layer
    keras.layers.Dense(units=1, activation='sigmoid')
])

# 3. Compile
optimized_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train with Early Stopping Callback
history_optimized = optimized_model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,             # Set a high epoch limit; early stopping will halt it early
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1
)

# 5. Evaluate the optimized model
loss_opt, accuracy_opt = optimized_model.evaluate(X_test, y_test)
print(f"\nOptimized Model - Test Loss: {loss_opt:.4f}, Test Accuracy: {accuracy_opt:.4f}")